## Gold Preprocessing (Deliverable 1.3.1)

This notebook completes the Gold-layer preprocessing stage of the Medallion Architecture. It prepares the pump sensor dataset for model training and for the comparative evaluation described in Section C of the project proposal.

**Purpose:**  
To transform the cleaned Silver-layer dataset into a fully model-ready Gold dataset using the final feature registry, imputation decisions, and anomaly labels produced during Silver EDA. This ensures that both the baseline model and the three-stage cascade model are trained on a consistent and reproducible feature set.

**Key Goals:**

- Load the finalized Silver dataset and feature registry.
- Apply the Silver EDA imputation strategy (forward-fill followed by median).
- Standardize and scale feature values as required for model stability.
- Construct the model-ready Gold dataframe with:
  - 50 vetted numeric features for Stage 1 (broad) modeling,
  - A reduced feature subset for Stage 2 (narrow) modeling,
  - Profile- and rule-based sensor groupings for Stage 3 confirmation logic.
- Generate and export all Gold-layer preprocessing artifacts, including:
  - The Gold preprocessed parquet dataset,
  - Stage 1 and Stage 2 feature sets,
  - Stage 3 rule/profile sensor lists,
  - A preprocessing summary and metadata record.

**Relevance to Section C:**  
Outputs from this notebook directly support the methods described in Section C by:

- Providing a stable, aligned feature matrix for the baseline Isolation Forest and the three-stage cascade (C.2, C.2.A).
- Ensuring consistent preprocessing necessary for the paired model comparison and alert-quality evaluation (C.4).
- Supplying the structured Gold dataset that underpins the visual communication of alert patterns and model differences (C.6).

This notebook finalizes the dataset that the Gold Modeling notebook will use to implement, evaluate, and compare both anomaly-detection approaches.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 


from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [2]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [3]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [4]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_preprocessing__v001_cascade"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
SILVER_FILE_NAME = f"{DATASET_NAME}__silver__train.parquet"

GOLD_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed.parquet"
GOLD_TRAIN_FILE_NAME = f"{DATASET_NAME}__gold__train.parquet"
GOLD_TEST_FILE_NAME = f"{DATASET_NAME}__gold__test.parquet"
GOLD_FIT_FILE_NAME = f"{DATASET_NAME}__gold__fit_normal_only.parquet"


FEATURE_REGISTRY_FILE_NAME = f"{DATASET_NAME}__silver__feature_registry.json"
IMPUTE_RECOMMENDATION_FILE_NAME = "imputation__recommendation.json"

STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"
STAGE2_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage2_features.json"
STAGE3_PRIMARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_primary_rule_sensors.json"
STAGE3_SECONDARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_secondary_rule_sensors.json"
CASCADE_RESULTS_FILE_NAME = f"{DATASET_NAME}__gold__cascade_results.csv"
COMPARISON_FILE_NAME = f"{DATASET_NAME}__gold__comparison__baseline_vs_cascade.csv"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Split configuration
TRAIN_FRACTION = 0.80
RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Scaling posture
USE_ROBUST_SCALER = False

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


# Stage 2 / Stage 3 sizing
STAGE2_TARGET_FEATURE_COUNT = 12
STAGE3_PRIMARY_COUNT = 5
STAGE3_SECONDARY_COUNT = 8

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 



GOLD_PREPROCESSING_LEDGER_FILE_NAME = f"ledger__{DATASET_NAME}__gold_preprocessing.json"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [6]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Paths

# Silver
SILVER_TRAIN_DATA_PATH = paths.data_silver_train / SILVER_FILE_NAME
#SILVER_ARTIFACTS_PATH = paths.artifacts / "silver" / DATASET_NAME
SILVER_ARTIFACTS_PATH = paths.artifacts / "silver" / DATASET_NAME
SILVER_EDA_ARTIFACTS_PATH = paths.artifacts / "silver_eda" / DATASET_NAME

# Gold
GOLD_DATA_PATH = paths.data_gold / GOLD_FILE_NAME

GOLD_TRAIN_DATA_PATH = paths.data_gold / GOLD_TRAIN_FILE_NAME
GOLD_TEST_DATA_PATH = paths.data_gold / GOLD_TEST_FILE_NAME
GOLD_FIT_DATA_PATH = paths.data_gold / GOLD_FIT_FILE_NAME
GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

# Jsons
FEATURE_REGISTRY_PATH = SILVER_ARTIFACTS_PATH / FEATURE_REGISTRY_FILE_NAME
IMPUTE_RECOMMENDATION_PATH = SILVER_EDA_ARTIFACTS_PATH / IMPUTE_RECOMMENDATION_FILE_NAME

# Stages
STAGE1_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE1_FEATURES_FILE_NAME
STAGE2_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE2_FEATURES_FILE_NAME
STAGE3_PRIMARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_PRIMARY_FILE_NAME
STAGE3_SECONDARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_SECONDARY_FILE_NAME
CASCADE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / CASCADE_RESULTS_FILE_NAME
COMPARISON_PATH = GOLD_ARTIFACTS_PATH / COMPARISON_FILE_NAME

# Preprocessing Summaries
PREPROCESSING_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__preprocessing_summary.json"
PREPROCESSING_METADATA_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__preprocessing_metadata.json"
REFERENCE_PROFILE_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__reference_profile.csv"
PREPROCESSOR_SCALER_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__preprocessor_scaler.joblib"


# Logs
LOGS_PATH = paths.logs

# Path Failsafes
GOLD_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)




In [8]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [9]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_preprocessing.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold stage starting")

# Log paths loads
log_gold_paths(paths, logger)


2026-03-02 01:17:08,144 | INFO | capstone.gold | Gold stage starting
2026-03-02 01:17:08,146 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-03-02 01:17:08,148 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-03-02 01:17:08,150 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-02 01:17:08,151 | INFO | capstone.gold | Project Notebooks Path Loaded /workspace/notebooks
2026-03-02 01:17:08,153 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-03-02 01:17:08,154 | INFO | capstone.gold | Data Bronze Path Loaded: /workspace/data/bronze
2026-03-02 01:17:08,156 | INFO | capstone.gold | Data Bronze Training Path Loaded: /workspace/data/bronze/train
2026-03-02 01:17:08,158 | INFO | capstone.gold | Data Bronze Testing Path Loaded: /workspace/data/bronze/test
2026-03-02 01:17:08,161 | INFO | capstone.gold | Data Silver Path Loaded: /workspace/data/silver
2026-03-02 01:17:08,162 | INFO | capstone.g

In [10]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [11]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_preprocessing",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "silver_path": str(SILVER_TRAIN_DATA_PATH),
        "feature_registry_path": str(FEATURE_REGISTRY_PATH),
        "impute_recommendation_path": str(IMPUTE_RECOMMENDATION_PATH),
        "gold_output_path": str(GOLD_DATA_PATH),
        "use_robust_scaler": bool(USE_ROBUST_SCALER),
        "stage2_target_feature_count": int(STAGE2_TARGET_FEATURE_COUNT),
        "stage3_primary_count": int(STAGE3_PRIMARY_COUNT),
        "stage3_secondary_count": int(STAGE3_SECONDARY_COUNT),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-03-02 01:17:12,606 | INFO | capstone.gold | W&B initialized: gold__001


In [12]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [13]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-03-02 01:17:13,444 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T01:17:13.444586+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-03-02T01:17:13.444586+00:00',
 'stage': 'gold',
 'recipe': 'gold_preprocessing__v001_cascade',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

In [14]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [15]:
silver_path = SILVER_TRAIN_DATA_PATH
feature_registry_path = FEATURE_REGISTRY_PATH
imputation_recommendation_path = IMPUTE_RECOMMENDATION_PATH

logger.info("Loading Silver parquet: %s", silver_path)
silver_dataframe = load_data(silver_path)

logger.info("Loading feature registry: %s", feature_registry_path)
feature_registry = load_json(feature_registry_path)
feature_columns = feature_registry.get("feature_columns", [])
feature_set_id = feature_registry.get("feature_set_id", "unknown_feature_set")

logger.info("Loading imputation recommendation: %s", imputation_recommendation_path)
imputation_recommendation = load_json(imputation_recommendation_path, raise_if_missing=False, default={})
recommended_imputation = imputation_recommendation.get("recommendation", "global_median")

logger.info("Silver shape=%s", silver_dataframe.shape)
logger.info("Feature count=%d", len(feature_columns))
logger.info("Recommended imputation=%s", recommended_imputation)

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="load_inputs",
    message="Loaded Silver parquet, feature registry, and imputation recommendation.",
    data={
        "silver_path": str(silver_path),
        "feature_registry_path": str(feature_registry_path),
        "impute_recommendation_path": str(imputation_recommendation_path),
        "feature_count": int(len(feature_columns)),
        "feature_set_id": str(feature_set_id),
        "recommended_imputation": str(recommended_imputation),
        "shape": {"rows": int(len(silver_dataframe)), "cols": int(len(silver_dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

silver_dataframe.head(3)

2026-03-02 01:17:14,047 | INFO | capstone.gold | Loading Silver parquet: /workspace/data/silver/train/pump__silver__train.parquet
2026-03-02 01:17:14,052 | INFO | capstone.file_io | Loading Parquet: /workspace/data/silver/train/pump__silver__train.parquet
2026-03-02 01:17:15,137 | INFO | capstone.gold | Loading feature registry: /workspace/artifacts/silver/pump/pump__silver__feature_registry.json
2026-03-02 01:17:15,143 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/silver/pump/pump__silver__feature_registry.json
2026-03-02 01:17:15,161 | INFO | capstone.gold | Loading imputation recommendation: /workspace/artifacts/silver_eda/pump/imputation__recommendation.json
2026-03-02 01:17:15,168 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/silver_eda/pump/imputation__recommendation.json
2026-03-02 01:17:15,184 | INFO | capstone.gold | Silver shape=(220320, 85)
2026-03-02 01:17:15,186 | INFO | capstone.gold | Feature count=50
2026-03-02 01:17:15,188 | INFO | cap

,meta__asset_id,meta__cleaning_recipe_id,meta__dataset,meta__dataset_name,meta__dataset_source,meta__event_id,meta__event_time_parse_success_percent,meta__event_time_source,meta__feature_count,meta__feature_set_id,meta__has_label_candidates,meta__has_status_candidates,meta__ingested_at_utc,meta__label_source,meta__label_source_column,meta__label_source_kind,meta__label_type,meta__layer,meta__processed_at_utc,meta__record_id,meta__run_id,meta__silver_version,meta__source_file,meta__source_row_id,meta__split,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status
0,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:0,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,14598431322315673869,run__001,silver__001,sensor.csv,0,unsplit,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:00:00,NORMAL
1,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:1,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,15954729095895098000,run__001,silver__001,sensor.csv,1,unsplit,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:01:00,NORMAL
2,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:2,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,10041703297090838359,run__001,silver__001,sensor.csv,2,unsplit,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.444734,47.35243,53.2118,46.39757,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,203.7037,2018-04-01 00:02:00,NORMAL


In [16]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [17]:
def build_time_ordered_split_mask(
    dataframe: pd.DataFrame,
    *,
    train_fraction: float,
) -> tuple[pd.Series, dict]:
    if "time_index" in dataframe.columns:
        ordering_series = dataframe["time_index"].astype("int64")
        ordering_column = "time_index"
    elif "event_step" in dataframe.columns:
        ordering_series = dataframe["event_step"].astype("int64")
        ordering_column = "event_step"
    else:
        ordering_series = pd.Series(np.arange(len(dataframe), dtype=np.int64), index=dataframe.index)
        ordering_column = "row_index"

    sorted_index = ordering_series.sort_values().index
    train_row_count = int(np.floor(len(sorted_index) * train_fraction))

    train_index_values = set(sorted_index[:train_row_count].tolist())
    train_mask = dataframe.index.to_series().apply(lambda index_value: index_value in train_index_values)

    split_info = {
        "train_fraction": float(train_fraction),
        "train_rows": int(train_mask.sum()),
        "test_rows": int((~train_mask).sum()),
        "ordering_column": ordering_column,
    }

    return train_mask, split_info

In [18]:
train_mask, split_info = build_time_ordered_split_mask(
    silver_dataframe,
    train_fraction=TRAIN_FRACTION,
)

ledger.add(
    kind="step",
    step="build_time_ordered_split",
    message="Created time-ordered train/test split for Gold modeling.",
    data=split_info,
    logger=logger,
)

split_info

2026-03-02 01:17:16,191 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T01:17:16.191251+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'build_time_ordered_split', 'message': 'Created time-ordered train/test split for Gold modeling.', 'why': None, 'consequence': None, 'data': {'train_fraction': 0.8, 'train_rows': 176256, 'test_rows': 44064, 'ordering_column': 'time_index'}}


{'train_fraction': 0.8,
 'train_rows': 176256,
 'test_rows': 44064,
 'ordering_column': 'time_index'}

In [19]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [20]:
def select_numeric_feature_columns(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> list[str]:
    numeric_feature_columns: list[str] = []

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns:
            continue
        if pd.api.types.is_numeric_dtype(dataframe[feature_name]):
            numeric_feature_columns.append(feature_name)

    return numeric_feature_columns


In [21]:
numeric_feature_columns = select_numeric_feature_columns(
    silver_dataframe,
    feature_columns=feature_columns,
)

In [22]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [23]:
def apply_imputation(
    dataframe: pd.DataFrame,
    *,
    numeric_feature_columns: list[str],
    method: str,
) -> tuple[pd.DataFrame, dict]:
    working_dataframe = dataframe.copy()

    if method == "forward_fill_within_group_then_median":
        grouping_columns: list[str] = []

        if "meta__asset_id" in working_dataframe.columns:
            grouping_columns.append("meta__asset_id")
        if "meta__run_id" in working_dataframe.columns:
            grouping_columns.append("meta__run_id")

        ordering_column = None
        if "event_step" in working_dataframe.columns:
            ordering_column = "event_step"
        elif "time_index" in working_dataframe.columns:
            ordering_column = "time_index"

        if len(grouping_columns) > 0 and ordering_column is not None:
            working_dataframe = working_dataframe.sort_values(
                grouping_columns + [ordering_column]
            ).reset_index(drop=True)

            for feature_name in numeric_feature_columns:
                working_dataframe[feature_name] = (
                    working_dataframe
                    .groupby(grouping_columns, dropna=False)[feature_name]
                    .ffill()
                )

        for feature_name in numeric_feature_columns:
            median_value = float(working_dataframe[feature_name].median(skipna=True))
            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(median_value)

        return working_dataframe, {
            "imputation_method": method,
            "grouping_columns": grouping_columns,
            "ordering_column": ordering_column,
        }

    if method == "global_mean":
        for feature_name in numeric_feature_columns:
            mean_value = float(working_dataframe[feature_name].mean(skipna=True))
            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(mean_value)

        return working_dataframe, {
            "imputation_method": method,
            "grouping_columns": [],
            "ordering_column": None,
        }

    for feature_name in numeric_feature_columns:
        median_value = float(working_dataframe[feature_name].median(skipna=True))
        working_dataframe[feature_name] = working_dataframe[feature_name].fillna(median_value)

    return working_dataframe, {
        "imputation_method": "global_median",
        "grouping_columns": [],
        "ordering_column": None,
    }


In [24]:
gold_working_dataframe = silver_dataframe.copy()

imputed_subset, imputation_info = apply_imputation(
    gold_working_dataframe.copy(),
    numeric_feature_columns=numeric_feature_columns,
    method=recommended_imputation,
)


In [25]:
gold_working_dataframe = imputed_subset.copy()

gold_working_dataframe["meta__layer"] = "gold"
gold_working_dataframe["meta__gold_imputation"] = imputation_info["imputation_method"]
gold_working_dataframe["meta__gold_scaler"] = "none"

scaler = None

if USE_ROBUST_SCALER:
    scaler = RobustScaler()
    scaled_values = scaler.fit_transform(gold_working_dataframe[numeric_feature_columns].values)
    scaled_feature_dataframe = pd.DataFrame(
        scaled_values,
        columns=numeric_feature_columns,
        index=gold_working_dataframe.index,
    )
    for feature_name in numeric_feature_columns:
        gold_working_dataframe[feature_name] = scaled_feature_dataframe[feature_name]
    gold_working_dataframe["meta__gold_scaler"] = "RobustScaler"

gold_build_info = {
    "numeric_feature_count": int(len(numeric_feature_columns)),
    "feature_set_id": str(feature_set_id),
    "imputation": gold_working_dataframe["meta__gold_imputation"].iloc[0],
    "scaler": gold_working_dataframe["meta__gold_scaler"].iloc[0],
}

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="build_gold_model_ready_dataframe",
    message="Built Gold model-ready dataframe using Silver feature registry and Silver EDA imputation recommendation.",
    data=gold_build_info,
    logger=logger,
)
#### #### #### #### #### #### #### #### 

gold_build_info

2026-03-02 01:17:23,113 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T01:17:23.113243+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'build_gold_model_ready_dataframe', 'message': 'Built Gold model-ready dataframe using Silver feature registry and Silver EDA imputation recommendation.', 'why': None, 'consequence': None, 'data': {'numeric_feature_count': 50, 'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40', 'imputation': 'forward_fill_within_group_then_median', 'scaler': 'none'}}


{'numeric_feature_count': 50,
 'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40',
 'imputation': 'forward_fill_within_group_then_median',
 'scaler': 'none'}

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
gold_preprocessed_path = save_data(
    gold_working_dataframe,
    GOLD_DATA_PATH.parent,
    GOLD_DATA_PATH.name,
)

wandb.save(str(gold_preprocessed_path))

ledger.add(
    kind="step",
    step="save_gold_preprocessed",
    message="Saved full Gold preprocessed dataset.",
    data={
        "gold_preprocessed_path": str(gold_preprocessed_path),
        "gold_shape": list(gold_working_dataframe.shape),
    },
    logger=logger,
)

In [26]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [27]:
def get_training_rows_for_unsupervised_model(
    dataframe: pd.DataFrame,
    *,
    train_mask: pd.Series,
) -> pd.DataFrame:
    training_subset = dataframe.loc[train_mask].copy()

    if "anomaly_flag" in training_subset.columns:
        training_subset = training_subset[training_subset["anomaly_flag"] == 0].copy()

    return training_subset

In [28]:
training_rows_for_fit = get_training_rows_for_unsupervised_model(
    gold_working_dataframe,
    train_mask=train_mask,
)

In [29]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [30]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

In [31]:
reference_profile = build_reference_profile(
    training_rows_for_fit,
    feature_columns=numeric_feature_columns,
)


In [32]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [33]:
def choose_stage2_features_from_training_stability(
    training_dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    target_count: int,
) -> list[str]:
    ranking_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = training_dataframe[feature_name]
        median_value = float(feature_series.median())
        standard_deviation = float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0

        coefficient_of_variation = standard_deviation / max(abs(median_value), 1e-6)

        ranking_rows.append({
            "feature_name": feature_name,
            "median_value": median_value,
            "standard_deviation": standard_deviation,
            "coefficient_of_variation": coefficient_of_variation,
        })

    ranking_dataframe = pd.DataFrame(ranking_rows)

    ranking_dataframe = ranking_dataframe.sort_values(
        by=["coefficient_of_variation", "standard_deviation", "feature_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    chosen_features = ranking_dataframe["feature_name"].head(target_count).tolist()
    return chosen_features

In [34]:
stage1_feature_columns = list(numeric_feature_columns)

stage2_feature_columns = choose_stage2_features_from_training_stability(
    training_rows_for_fit,
    feature_columns=stage1_feature_columns,
    target_count=STAGE2_TARGET_FEATURE_COUNT,
)


In [35]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [36]:
def build_stage3_sensor_groups(
    reference_profile: pd.DataFrame,
    *,
    stage2_feature_columns: list[str],
    primary_count: int,
    secondary_count: int,
) -> tuple[list[str], list[str]]:
    ranked_reference = reference_profile[reference_profile["feature_name"].isin(stage2_feature_columns)].copy()

    ranked_reference = ranked_reference.sort_values(
        by=["standard_deviation", "feature_name"],
        ascending=[True, True]
    ).reset_index(drop=True)

    primary_rule_sensors = ranked_reference["feature_name"].head(primary_count).tolist()

    remaining_features = [
        feature_name
        for feature_name in ranked_reference["feature_name"].tolist()
        if feature_name not in primary_rule_sensors
    ]

    secondary_rule_sensors = remaining_features[:secondary_count]

    return primary_rule_sensors, secondary_rule_sensors

In [37]:
stage3_primary_rule_sensors, stage3_secondary_rule_sensors = build_stage3_sensor_groups(
    reference_profile,
    stage2_feature_columns=stage2_feature_columns,
    primary_count=STAGE3_PRIMARY_COUNT,
    secondary_count=STAGE3_SECONDARY_COUNT,
)

In [38]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [39]:
save_json(stage1_feature_columns, STAGE1_FEATURES_PATH)
save_json(stage2_feature_columns, STAGE2_FEATURES_PATH)
save_json(stage3_primary_rule_sensors, STAGE3_PRIMARY_PATH)
save_json(stage3_secondary_rule_sensors, STAGE3_SECONDARY_PATH)

wandb.save(str(STAGE1_FEATURES_PATH))
wandb.save(str(STAGE2_FEATURES_PATH))
wandb.save(str(STAGE3_PRIMARY_PATH))
wandb.save(str(STAGE3_SECONDARY_PATH))

feature_set_summary = {
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
}

ledger.add(
    kind="step",
    step="build_stage_feature_sets",
    message="Built feature sets for baseline, Stage 1, Stage 2, and Stage 3.",
    data=feature_set_summary,
    logger=logger,
)

feature_set_summary

2026-03-02 01:17:29,143 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-02 01:17:29,178 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage2_features.json
2026-03-02 01:17:29,205 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage3_primary_rule_sensors.json
2026-03-02 01:17:29,233 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage3_secondary_rule_sensors.json
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
2026-03-02 01:17:29,393 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T01:17:29.393328+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'build_stage_feature_sets', 'message': 'Built feature sets for baseline, Stage 1, Stage 2, and Stag

{'stage1_feature_count': 50,
 'stage2_feature_count': 12,
 'stage3_primary_rule_count': 5,
 'stage3_secondary_rule_count': 7}

In [40]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

## Save Gold Split Artifacts

This step saves the Gold preprocessing outputs as separate parquet artifacts so downstream modeling notebooks can use a fixed and reproducible split.

The notebook already created a time-ordered split earlier in the workflow:

- The **train split** contains the earlier portion of the time-ordered Gold dataset.
- The **test split** contains the later holdout portion of the time-ordered Gold dataset.
- The **fit subset** contains only normal rows from the training split and is used for unsupervised model fitting.

Saving these artifacts ensures that the baseline, cascade, and comparison notebooks all use the same consistent partitions.

In [ ]:
gold_train_dataframe = gold_working_dataframe.loc[train_mask].copy()
gold_test_dataframe = gold_working_dataframe.loc[~train_mask].copy()
gold_fit_dataframe = training_rows_for_fit.copy()



gold_train_path = save_data(
    gold_train_dataframe,
    GOLD_TRAIN_DATA_PATH.parent,
    GOLD_TRAIN_DATA_PATH.name,
)

gold_test_path = save_data(
    gold_test_dataframe,
    GOLD_TEST_DATA_PATH.parent,
    GOLD_TEST_DATA_PATH.name,
)

gold_fit_path = save_data(
    gold_fit_dataframe,
    GOLD_FIT_DATA_PATH.parent,
    GOLD_FIT_DATA_PATH.name,
)

wandb.save(str(gold_train_path))
wandb.save(str(gold_test_path))
wandb.save(str(gold_fit_path))

gold_split_summary = {
    "train_rows": int(len(gold_train_dataframe)),
    "test_rows": int(len(gold_test_dataframe)),
    "fit_rows_normal_only": int(len(gold_fit_dataframe)),
    "train_path": str(gold_train_path),
    "test_path": str(gold_test_path),
    "fit_path": str(gold_fit_path),
}

ledger.add(
    kind="step",
    step="save_gold_split_artifacts",
    message="Saved Gold train, test, and normal-only fit parquet artifacts.",
    data=gold_split_summary,
    logger=logger,
)

pd.DataFrame(
    [
        {
            "split": "train",
            "rows": int(len(gold_train_dataframe)),
            "abnormal_rows": int(gold_train_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_train_dataframe.columns else None,
            "path": str(gold_train_path),
        },
        {
            "split": "test",
            "rows": int(len(gold_test_dataframe)),
            "abnormal_rows": int(gold_test_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_test_dataframe.columns else None,
            "path": str(gold_test_path),
        },
        {
            "split": "fit_normal_only",
            "rows": int(len(gold_fit_dataframe)),
            "abnormal_rows": int(gold_fit_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_fit_dataframe.columns else None,
            "path": str(gold_fit_path),
        },
    ]
)

2026-03-02 01:17:30,635 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/data/gold/pump__gold__train.parquet
2026-03-02 01:17:32,660 | INFO | capstone.file_io | Saved: pump__gold__train.parquet | shape=(176256, 87) | columns=['meta__asset_id', 'meta__cleaning_recipe_id', 'meta__dataset', 'meta__dataset_name', 'meta__dataset_source', 'meta__event_id', 'meta__event_time_parse_success_percent', 'meta__event_time_source', 'meta__feature_count', 'meta__feature_set_id', 'meta__has_label_candidates', 'meta__has_status_candidates', 'meta__ingested_at_utc', 'meta__label_source', 'meta__label_source_column', 'meta__label_source_kind', 'meta__label_type', 'meta__layer', 'meta__processed_at_utc', 'meta__record_id', 'meta__run_id', 'meta__silver_version', 'meta__source_file', 'meta__source_row_id', 'meta__split', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_

,split,rows,abnormal_rows,path
0,train,176256,14484,/workspace/data/gold/pump__gold__train.parquet
1,test,44064,0,/workspace/data/gold/pump__gold__test.parquet
2,fit_normal_only,161772,0,/workspace/data/gold/pump__gold__fit_normal_on...


In [ ]:
scaler = locals().get("scaler", None)

preprocessing_summary = {
    "gold_path": str(GOLD_DATA_PATH),
    "gold_shape": list(gold_working_dataframe.shape),
    "feature_count": int(len(numeric_feature_columns)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "imputation_method": str(imputation_info.get("imputation_method")),
    "grouping_columns": list(imputation_info.get("grouping_columns", [])),
    "ordering_column": imputation_info.get("ordering_column"),
    "scaler_used": bool(USE_ROBUST_SCALER),
    "train_fraction": float(TRAIN_FRACTION),
    "train_rows": int(len(gold_train_dataframe)),
    "test_rows": int(len(gold_test_dataframe)),
    "fit_rows_normal_only": int(len(gold_fit_dataframe)),
    "gold_train_path": str(gold_train_path),
    "gold_test_path": str(gold_test_path),
    "gold_fit_path": str(gold_fit_path),
}

save_json(preprocessing_summary, PREPROCESSING_SUMMARY_PATH)

preprocessing_metadata = {
    "recipe_id": RECIPE_ID,
    "gold_version": GOLD_VERSION,
    "dataset_name": DATASET_NAME,
    "feature_set_source": str(FEATURE_REGISTRY_PATH),
    "imputation_recommendation_source": str(IMPUTE_RECOMMENDATION_PATH),
    "gold_output_path": str(GOLD_DATA_PATH),
    "reference_profile_path": str(REFERENCE_PROFILE_PATH),
    "stage1_features_path": str(STAGE1_FEATURES_PATH),
    "stage2_features_path": str(STAGE2_FEATURES_PATH),
    "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
    "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    "preprocessor_scaler_path": str(PREPROCESSOR_SCALER_PATH) if USE_ROBUST_SCALER else None,
    "gold_train_path": str(gold_train_path),
    "gold_test_path": str(gold_test_path),
    "gold_fit_path": str(gold_fit_path),
}

save_json(preprocessing_metadata, PREPROCESSING_METADATA_PATH)
reference_profile.to_csv(REFERENCE_PROFILE_PATH, index=False)

if USE_ROBUST_SCALER and scaler is not None:
    joblib.dump(scaler, PREPROCESSOR_SCALER_PATH)

wandb.save(str(PREPROCESSING_SUMMARY_PATH))
wandb.save(str(PREPROCESSING_METADATA_PATH))
wandb.save(str(REFERENCE_PROFILE_PATH))

if USE_ROBUST_SCALER and scaler is not None:
    wandb.save(str(PREPROCESSOR_SCALER_PATH))

ledger.add(
    kind="step",
    step="save_preprocessing_outputs",
    message="Saved Gold preprocessing outputs, metadata, and reference profile artifacts.",
    data={
        "preprocessing_summary_path": str(PREPROCESSING_SUMMARY_PATH),
        "preprocessing_metadata_path": str(PREPROCESSING_METADATA_PATH),
        "reference_profile_path": str(REFERENCE_PROFILE_PATH),
        "preprocessor_scaler_path": str(PREPROCESSOR_SCALER_PATH) if USE_ROBUST_SCALER and scaler is not None else None,
    },
    logger=logger,
)

ledger.add(
    kind="step",
    step="finalize_preprocessing",
    message="Gold preprocessing notebook complete.",
    data={
        "gold_path": str(GOLD_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_shape": list(gold_working_dataframe.shape),
        "feature_count": int(len(numeric_feature_columns)),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

preprocessing_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_PREPROCESSING_LEDGER_FILE_NAME
ledger.write_json(preprocessing_ledger_path)

wandb.save(str(preprocessing_ledger_path))
wandb_run.finish()

2026-03-02 01:19:15,355 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__preprocessing_summary.json
2026-03-02 01:19:15,385 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__preprocessing_metadata.json


2026-03-02 01:19:15,597 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T01:19:15.597195+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'save_preprocessing_outputs', 'message': 'Saved Gold preprocessing outputs, metadata, and reference profile artifacts.', 'why': None, 'consequence': None, 'data': {'preprocessing_summary_path': '/workspace/artifacts/gold/pump/pump__gold__preprocessing_summary.json', 'preprocessing_metadata_path': '/workspace/artifacts/gold/pump/pump__gold__preprocessing_metadata.json', 'reference_profile_path': '/workspace/artifacts/gold/pump/pump__gold__reference_profile.csv', 'preprocessor_scaler_path': None}}
2026-03-02 01:19:15,599 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T01:19:15.599149+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'finalize_preprocessing', 'message': 'Gold preprocessing notebook complete.', 'why': None, 'consequence': None, 'da